# 第 14 章习题与解答

## Exercise 14.1

**题目**:画一个 2 轮 tool-call 轨迹的时序图(user → assistant tool_call → tool_response → assistant final answer),标注哪些 token 的 response_mask=1。

<details><summary><b>参考答案</b></summary>

```
[response_mask=0] <|im_start|>user\n北京天气?<|im_end|>\n
[response_mask=0] <|im_start|>assistant\n
[response_mask=1] <tool_call>{"name":"weather","city":"北京"}</tool_call>
[response_mask=0] <|im_end|>\n
[response_mask=0] <tool_response>{"temp":25}</tool_response>\n
[response_mask=0] <|im_start|>assistant\n
[response_mask=1] 北京今天气温 25 度。
[response_mask=0] <|im_end|>
```

- mask=1 的部分:模型生成的 `<tool_call>` 和最终回复 —— 参与 RL loss
- mask=0 的部分:user prompt、`<tool_response>`(外部注入)、特殊 token —— 不参与

</details>

## Exercise 14.2

**题目**:为什么用 mock 工具而非真实 API?如果用真实 API 训练会有什么问题?

<details><summary><b>参考答案</b></summary>

用真实 API 的问题:

1. **不可复现**:同一个 prompt,API 可能返回不同结果(天气 API 的数据会变)。RL 需要确定性来计算 old_logps 和 ratio —— 如果环境变化,old_logps 失效,ratio 不可信。
2. **延迟和费用**:RL 训练需要成千上万次 rollout。真实 API 的延迟(每次几百 ms)和费用会让训练不可行。
3. **奖励噪声**:真实 API 返回的数据有噪声(网络波动、数据质量不一),导致 reward 方差极大,训练不稳定。
4. **安全风险**:训练时频繁调用真实 API 可能触发限流、泄露 API key。

Mock 工具保证了:确定性(可复现)、零延迟、零成本、无噪声。

</details>

## Exercise 14.3

**题目**:延迟奖励 vs 即时奖励:在多轮 tool-use 场景中,为什么延迟奖励更合理?如果每步都给 reward 会怎样?

<details><summary><b>参考答案</b></summary>

**延迟奖励更合理的原因**:

中间步骤的价值取决于后续结果。例如:
- Round 1 调用了 `weather` 工具 → 如果 Round 2 利用结果给出了正确答案,Round 1 的调用是好的
- 同样的调用 → 如果 Round 2 完全忽略结果,Round 1 的调用就是浪费

只有在轨迹结束后,才能评判中间步骤的好坏。

**如果每步都给 reward**:

- 需要**为每个工具调用单独定义奖励标准**(这个调用「好不好」?)—— 这非常难定义
- 可能导致**局部最优**:模型学会频繁调用工具(因为调用本身就有 reward),但不关心结果
- 增加 reward shaping 的复杂度(怎么平衡每步的 reward?)

> GRPO 的简洁做法:整条轨迹一个 reward,靠组归一化分配信号。模型自己学会「哪些调用是值得的」。

</details>